# المحاضرة 40: التقطيع الدلالي الجاهز باستخدام مكتبة LangChain
### Built-in Semantic Chunking with LangChain & Threshold Strategies

> **التوثيق والشرح النظري والمعماري الشامل:**
> راجع ملف التوثيق المخصص في مجلد الشرح: [`docs/04-advanced-chunking/04-semantic-chunking-langchain.md`](../../../docs/04-advanced-chunking/04-semantic-chunking-langchain.md)

---

## الكود النموذجي المرجعي (Production Reference Code)
يتضمن هذا الدفتر التطبيق العملي الكامل لأداة `SemanticChunker` المدمجة في LangChain، مع مقارنة استراتيجيات العتبات الإحصائية وبناء مسار RAG متكامل وفق التكليف الوارد في المحاضرة.


In [1]:
# 1. استدعاء المكتبات وتهيئة متغيرات البيئة
import os
import warnings
from dotenv import load_dotenv

# Suppress minor deprecation warnings for clean execution logs
warnings.filterwarnings("ignore")
load_dotenv()

# Document loading and embeddings
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# Vector store and retrieval
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

has_groq = bool(os.getenv("GROQ_API_KEY"))
print(f"[CONFIG] Environment loaded. GROQ_API_KEY available: {has_groq}")

[CONFIG] Environment loaded. GROQ_API_KEY available: True


In [2]:
# 2. تحميل المستند الخام باستخدام TextLoader
loader = TextLoader("langchain_intro.txt")
docs = loader.load()

print(f"[INFO] Loaded document count: {len(docs)}")
print(f"[INFO] Document source: {docs[0].metadata.get('source')}")
print(f"[INFO] Document character length: {len(docs[0].page_content)}\n")
print("--- Document Content Preview ---")
print(docs[0].page_content)

[INFO] Loaded document count: 1
[INFO] Document source: langchain_intro.txt
[INFO] Document character length: 287

--- Document Content Preview ---
LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.


In [3]:
# 3. تهيئة موديل التضمين ومحرك التقطيع الدلالي الافتراضي (Percentile)
# Initialize lightweight local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Initialize LangChain built-in SemanticChunker (defaults to 95th percentile threshold)
chunker_default = SemanticChunker(embeddings=embeddings)

# Split loaded documents
chunks_default = chunker_default.split_documents(docs)

print(f"[INFO] Default Percentile Semantic Chunker produced {len(chunks_default)} chunks:\n")
for idx, chunk in enumerate(chunks_default, 1):
    print(f"--- Chunk [{idx}] ({len(chunk.page_content)} characters) ---")
    print(f"{chunk.page_content}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Default Percentile Semantic Chunker produced 2 chunks:

--- Chunk [1] (208 characters) ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- Chunk [2] (78 characters) ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [4]:
# 4. مقارنة استراتيجيات العتبات الإحصائية (Breakpoint Threshold Types)
strategies = [
    ("percentile", 95),
    ("standard_deviation", 1.25),
    ("interquartile", 1.5)
]

print("[INFO] Evaluating LangChain Breakpoint Threshold Strategies:\n")
strategy_results = {}

for strategy_type, amount in strategies:
    custom_chunker = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type=strategy_type,
        breakpoint_threshold_amount=amount
    )
    result_chunks = custom_chunker.split_documents(docs)
    strategy_results[strategy_type] = result_chunks
    print(f"Strategy: '{strategy_type}' (amount={amount}) -> Produced {len(result_chunks)} chunks.")
    for idx, c in enumerate(result_chunks, 1):
        print(f"   [{idx}] {c.page_content[:60]}...")
    print("-" * 50)

[INFO] Evaluating LangChain Breakpoint Threshold Strategies:

Strategy: 'percentile' (amount=95) -> Produced 2 chunks.
   [1] LangChain is a framework for building applications with LLMs...
   [2] The Eiffel Tower is located in Paris. France is a popular to...
--------------------------------------------------
Strategy: 'standard_deviation' (amount=1.25) -> Produced 2 chunks.
   [1] LangChain is a framework for building applications with LLMs...
   [2] The Eiffel Tower is located in Paris. France is a popular to...
--------------------------------------------------
Strategy: 'interquartile' (amount=1.5) -> Produced 1 chunks.
   [1] LangChain is a framework for building applications with LLMs...
--------------------------------------------------


In [5]:
# 5. إنجاز تكليف المحاضرة: بناء مسار RAG كامل عبر LCEL مع FAISS و Groq
# Step A: Ingest semantic chunks into FAISS vector store
vector_store = FAISS.from_documents(chunks_default, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Step B: Define strict context-bound prompt template
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context:
{context}

Question:
{question}

Answer:""")

# Step C: Initialize Groq LLM
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

# Step D: Construct LCEL pipeline
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("[INFO] LCEL RAG pipeline assembled successfully.")

[INFO] LCEL RAG pipeline assembled successfully.


In [6]:
# 6. تنفيذ استعلامات التحقق وفحص عزل السياق
# Query 1: LangChain tools query
query_1 = "What tools can LangChain be combined with?"
response_1 = rag_chain.invoke({"question": query_1})

print(f"[QUERY 1]: {query_1}")
print(f"[RESPONSE 1]:\n{response_1}\n")
print("=" * 60)

# Query 2: Geography query
query_2 = "Where is the Eiffel Tower located?"
response_2 = rag_chain.invoke({"question": query_2})

print(f"[QUERY 2]: {query_2}")
print(f"[RESPONSE 2]:\n{response_2}")

[QUERY 1]: What tools can LangChain be combined with?
[RESPONSE 1]:
LangChain can be combined with tools such as **OpenAI** and **Pinecone**.



[QUERY 2]: Where is the Eiffel Tower located?
[RESPONSE 2]:
The Eiffel Tower is located in Paris, France.


In [7]:
# 7. التحقق البرمجي الصارم (Assertions)
assert len(chunks_default) == 2, "Default SemanticChunker must yield exactly 2 thematic chunks"
assert "OpenAI" in chunks_default[0].page_content, "First chunk must contain AI framework information"
assert "Paris" in chunks_default[1].page_content, "Second chunk must contain European geography"
assert "paris" in response_2.lower(), "Response 2 must correctly identify Paris"

print("[SUCCESS] All programmatic assertions passed successfully.")

[SUCCESS] All programmatic assertions passed successfully.


---
## مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص مخصصة، أو اختبار أنواع عتبات مختلفة (مثل gradient)، أو طرح استعلامات جديدة على مسار الـ RAG.

In [8]:
# مساحة التجربة والتطبيق العملي:
# جرب استدعاء SemanticChunker على مستند خاص بك واختبر استرجاع الإجابات هنا:

